# NHANES 2021–2023 – Bước 5: Huấn luyện mô hình cơ sở (baseline)

**Người chạy:** Bạn, trên Jupyter Notebook. **Notebook này chưa được thực thi hoặc train.**

⚠️ **CÓ CODE HUẤN LUYỆN.** Cell bắt đầu bằng `FIT / TRAIN` sẽ thật sự chạy mô hình khi **bạn** bấm `Shift + Enter`. Đọc toàn bộ phần hướng dẫn trước khi chạy.

**Mục tiêu hẹp:** Phân loại dấu hiệu `HbA1c ≥ 5,7%` ở người từ 20 tuổi tự khai chưa từng được bác sĩ thông báo mắc tiểu đường. **Không** phải xác suất mắc type 2 trong tương lai, không được chẩn đoán, không thay thế bác sĩ.

**Dữ liệu:** file Bước 4 `data/processed/nhanes_step4_screening_candidate_REVIEW_ONLY.csv`. Tập `screening_candidate` chưa giải quyết hết tình trạng mang thai không công bố. Báo cáo này thuộc loại nghiên cứu minh họa, chưa cân bằng survey weights.

Nhấn `Kernel → Restart Kernel and Clear All Outputs` nếu bạn đã chạy thử lộn thứ tự; sau đó chạy từng cell từ trên xuống.

## 0. Kiểm tra package

Nếu máy chưa có scikit-learn/joblib, chạy trong một cell Jupyter riêng: `%pip install "scikit-learn>=1.6" "joblib>=1.4"`. Sau đó restart kernel. Giữ Python 3.13.9 và pandas 3.0.0 đang có.

In [1]:
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
import joblib
from IPython.display import display

cwd = Path.cwd().resolve()
if (cwd / "data" / "processed").is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data" / "processed").is_dir():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Mở Jupyter từ 03-ML-Research hoặc 03-ML-Research/notebooks.")

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "nhanes_step4_screening_candidate_REVIEW_ONLY.csv"
RUN_DIR = PROJECT_ROOT / "data" / "outputs" / "model_training"
MODEL_DIR = PROJECT_ROOT / "models" / "baseline"
RUN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version(), "| pandas:", pd.__version__, "| sklearn:", sklearn.__version__)
print("Dataset:", DATA_FILE)

Python: 3.13.9 | pandas: 3.0.0 | sklearn: 1.8.0
Dataset: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\processed\nhanes_step4_screening_candidate_REVIEW_ONLY.csv


## 1. Kiểm tra dữ liệu đầu vào và chống rò rỉ nhãn

Dữ liệu phải được người dùng tự xuất từ Bước 4. Nếu thiếu file, **không** tự thay bằng bảng khác. Các số 4.698/1.387/3.311 là đối chiếu với kết quả Bước 4 đã gửi, không được sửa dữ liệu để khớp chúng.

In [2]:
if not DATA_FILE.exists():
    raise FileNotFoundError(f"Thiếu {DATA_FILE}. Chạy lại ô xuất file trong notebook Bước 4.")

df = pd.read_csv(DATA_FILE)
FEATURES = ["RIDAGEYR", "RIAGENDR", "BMXBMI"]
TARGET = "a1c_ge_57_candidate"
expected = {"SEQN", *FEATURES, TARGET}
assert expected.issubset(df.columns), f"Thiếu cột: {expected - set(df.columns)}"
assert df["SEQN"].notna().all() and df["SEQN"].is_unique, "SEQN thiếu/trùng"
assert df[TARGET].notna().all() and set(df[TARGET].unique()) <= {0, 1}, "Nhãn phải là 0/1, không thiếu"
assert set(df["RIAGENDR"].dropna().unique()) <= {1, 2}, "Mã RIAGENDR ngoài 1, 2"
FORBIDDEN = {"SEQN", "LBXGH", "DIQ010", "DIQ160", "RIDEXPRG", "WTPH2YR", "SDMVSTRA", "SDMVPSU"}
assert not (set(FEATURES) & FORBIDDEN)
print("Số người:", len(df))
print("Nhãn 0/1:")
display(df[TARGET].value_counts(dropna=False).sort_index().to_frame("n"))
print("Thiếu đặc trưng:")
display(df[FEATURES].isna().sum().to_frame("missing_n"))
print("Không dùng SEQN hoặc LBXGH trong X.")
if len(df) != 4698 or int(df[TARGET].sum()) != 1387:
    raise ValueError("Dataset khác kết quả Bước 4 (4698, 1387 nhãn 1); dừng và kiểm tra lại nguồn.")

Số người: 4698
Nhãn 0/1:


,n
a1c_ge_57_candidate,
0,3311
1,1387


Thiếu đặc trưng:


,missing_n
RIDAGEYR,0
RIAGENDR,0
BMXBMI,57


Không dùng SEQN hoặc LBXGH trong X.


## 2. Chia train / validation / test, giữ test làm holdout

- Train 70%, validation 15%, test 15%, `random_state=42`.
- Dùng `stratify` để giữ gần như cùng tỷ lệ nhãn 1 qua các tập.
- **KHÔNG** dùng test để chọn feature, tune model, chọn threshold hay báo metrics trong Bước 5.

In [3]:
from sklearn.model_selection import train_test_split

SEED = 42
trainval_df, test_df = train_test_split(
    df, test_size=0.15, random_state=SEED, stratify=df[TARGET]
)
train_df, val_df = train_test_split(
    trainval_df, test_size=0.15 / 0.85, random_state=SEED,
    stratify=trainval_df[TARGET]
)
assert len(df) == len(train_df) + len(val_df) + len(test_df)
assert set(train_df.SEQN).isdisjoint(val_df.SEQN)
assert set(train_df.SEQN).isdisjoint(test_df.SEQN)
assert set(val_df.SEQN).isdisjoint(test_df.SEQN)
X_train, y_train = train_df[FEATURES], train_df[TARGET].astype(int)
X_val, y_val = val_df[FEATURES], val_df[TARGET].astype(int)
print("Train:", len(train_df), "| Validation:", len(val_df), "| Test (KHÔNG ĐÁNH GIÁ Ở BƯỚC 5):", len(test_df))
print("Dự kiến tổng:", len(df))
# Không in nhãn/metric của test để tránh sử dụng holdout vào quyết định.

Train: 3288 | Validation: 705 | Test (KHÔNG ĐÁNH GIÁ Ở BƯỚC 5): 705
Dự kiến tổng: 4698


## 3. Tạo preprocessing pipeline đúng quy trình

`RIDAGEYR`, `BMXBMI`: điền thiếu bằng trung vị **học trên train**, sau đó standardize. `RIAGENDR`: điền frequent category và one-hot encoding. Không dùng mã giới tính như biến liên tục có khoảng cách định lượng.

Các bước tiền xử lý được gói trong pipeline, **không fit trước khi train**.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

numeric_features = ["RIDAGEYR", "BMXBMI"]
category_features = ["RIAGENDR"]

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

def make_preprocessor():
    return ColumnTransformer([
        ("num", num_pipe, numeric_features),
        ("cat", cat_pipe, category_features),
    ], remainder="drop")

models = {
    "Dummy": Pipeline([
        ("prep", make_preprocessor()),
        ("model", DummyClassifier(strategy="prior")),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", make_preprocessor()),
        ("model", LogisticRegression(max_iter=1500, random_state=SEED)),
    ]),
    "RandomForest": Pipeline([
        ("prep", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=250, max_depth=8, min_samples_leaf=10,
            random_state=SEED, n_jobs=-1
        )),
    ]),
}
print("Đã khởi tạo 3 Pipeline. Chưa fit/huấn luyện.")

Đã khởi tạo 3 Pipeline. Chưa fit/huấn luyện.


---

## 4. ⚠️ TỪ CELL TIẾP THEO LÀ HUẤN LUYỆN THẬT

**CHỈ BẠN MỚI CHẠY CELL BÊN DƯỚI.** Sau khi chạy sẽ `.fit()` ba pipeline trên training set. Dự án chưa train trước thời điểm này. Nếu muốn đổi cohort hoặc thêm feature, hãy dừng và điều chỉnh quyết định nghiên cứu trước.

**Không upload dữ liệu cá nhân thực tế vào các dịch vụ bên ngoài.** Bộ NHANES công khai nhưng vẫn cần tôn trọng quy tắc sử dụng/công bố.

In [5]:
# ===== FIT / TRAIN: BẠN CHỦ ĐỘNG CHẠY CELL NÀY =====
from time import perf_counter

trained = {}
train_times = {}
for name, estimator in models.items():
    start = perf_counter()
    estimator.fit(X_train, y_train)  # Imputer/Scaler/OneHot fit CHỈ trên train
    trained[name] = estimator
    train_times[name] = round(perf_counter() - start, 3)
    print(f"Đã train {name}: {train_times[name]:.3f}s")
print("Hoàn tất train baseline. Chưa dùng test set.")

Đã train Dummy: 0.036s
Đã train LogisticRegression: 0.041s
Đã train RandomForest: 0.907s
Hoàn tất train baseline. Chưa dùng test set.


## 5. Đánh giá **validation** (không phải final test)

Đánh giá tại threshold mặc định 0,5 chỉ để so sánh sơ bộ. Bước 6 sẽ xem calibration, threshold và holdout test cuối. Không chọn threshold bằng test.

In [6]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    confusion_matrix,
)

metrics_rows = []
validation_confusions = {}
for name, estimator in trained.items():
    p1 = estimator.predict_proba(X_val)[:, 1]
    pred = (p1 >= 0.5).astype(int)
    cm = confusion_matrix(y_val, pred, labels=[0, 1])
    validation_confusions[name] = cm.tolist()
    metrics_rows.append({
        "model": name,
        "roc_auc_val": roc_auc_score(y_val, p1),
        "pr_auc_val": average_precision_score(y_val, p1),
        "precision_val": precision_score(y_val, pred, zero_division=0),
        "recall_val": recall_score(y_val, pred, zero_division=0),
        "f1_val": f1_score(y_val, pred, zero_division=0),
        "accuracy_val": accuracy_score(y_val, pred),
        "brier_val": brier_score_loss(y_val, p1),
        "tn": int(cm[0,0]), "fp": int(cm[0,1]),
        "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        "train_seconds": train_times[name],
    })
results = pd.DataFrame(metrics_rows).sort_values(
    ["roc_auc_val", "brier_val"], ascending=[False, True]
).reset_index(drop=True)
display(results.round(4))
print("Chỉ là metric validation, CHƯA kiểm định test, CHƯA có giá trị lâm sàng.")

,model,roc_auc_val,pr_auc_val,precision_val,recall_val,f1_val,accuracy_val,brier_val,tn,fp,fn,tp,train_seconds
0,LogisticRegression,0.7173,0.4415,0.4416,0.1635,0.2386,0.6922,0.1868,454,43,174,34,0.041
1,RandomForest,0.7128,0.4516,0.4886,0.2067,0.2905,0.7021,0.1854,452,45,165,43,0.907
2,Dummy,0.5000,0.2950,0.0000,0.0000,0.0000,0.7050,0.2080,497,0,208,0,0.036


Chỉ là metric validation, CHƯA kiểm định test, CHƯA có giá trị lâm sàng.


## 6. Chọn mô hình theo validation và lưu artifact

**Quy tắc lựa chọn nghiên cứu baseline:** ROC-AUC validation cao nhất, hòa thì Brier thấp hơn. Quy tắc này chỉ giúp chốt một pipeline để kiểm thử holdout ở Bước 6; không phải lựa chọn lâm sàng tối ưu.

Các file `test_holdout_DO_NOT_TUNE.csv` phải giữ nguyên và KHÔNG dùng để tune cho đến Bước 6. Nếu bạn đã nhìn/test vào holdout để cải thiện model, phải thiết kế lại phép đánh giá và không công bố nó như independent test.

In [7]:
selected_name = str(results.loc[0, "model"])
selected_model = trained[selected_name]
print("Selected by VALIDATION ROC-AUC:", selected_name)

# Chỉ lưu X,y thử nghiệm cho Bước 6, không chạy dự đoán test ở đây.
for subset, f in [("train", train_df), ("validation", val_df), ("test_holdout_DO_NOT_TUNE", test_df)]:
    f.to_csv(RUN_DIR / f"{subset}.csv", index=False)
results.to_csv(RUN_DIR / "step5_validation_metrics.csv", index=False)
model_path = MODEL_DIR / "nhanes_hba1c_baseline_step5.joblib"
joblib.dump(selected_model, model_path)

metadata = {
    "phase": "phase3_step5",
    "source": "NHANES 2021-2023",
    "cohort": "screening_candidate self-report never diagnosed + exclude RIDEXPRG=1",
    "label_definition": "LBXGH >= 5.7% (not a diabetes diagnosis or future risk)",
    "features": FEATURES,
    "training_sample_count": int(len(train_df)),
    "validation_sample_count": int(len(val_df)),
    "test_holdout_sample_count": int(len(test_df)),
    "random_state": SEED,
    "selected_model": selected_name,
    "selection_criterion": "ROC-AUC on validation; Brier tiebreaker",
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "trained_by": "user locally in Jupyter Notebook",
    "test_metrics": None,
    "restrictions": [
        "research only; no diagnosis or clinical use",
        "not externally validated",
        "not weighted for NHANES complex survey design",
        "pregnancy missingness unresolved",
        "not a forecast of type 2 diabetes incidence",
    ],
}
(RUN_DIR / "step5_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved selected model:", model_path)
print("Saved validation metrics and split files:", RUN_DIR)

Selected by VALIDATION ROC-AUC: LogisticRegression
Saved selected model: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\models\baseline\nhanes_hba1c_baseline_step5.joblib
Saved validation metrics and split files: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\data\outputs\model_training


## 7. Output cuối để gửi lại ChatGPT

Chỉ gửi **kết quả tổng hợp, metric và lỗi**, không cần gửi nguyên dữ liệu hàng người tham gia. Tôi sẽ cập nhật tài liệu `05-Model-Training.md` và `09-Experiment-Log.md` từ kết quả thật, rồi hướng dẫn Bước 6 – Model Evaluation.

In [8]:
print("===== BƯỚC 5 — NHANES MODEL TRAINING (NGƯỜI DÙNG TỰ CHẠY) =====")
print("Python:", platform.python_version(), "| Pandas:", pd.__version__, "| sklearn:", sklearn.__version__)
print("Dataset:", len(df), "| Target 1:", int(df[TARGET].sum()), "| Target 0:", int((df[TARGET] == 0).sum()))
print("Train:", len(train_df), "| Validation:", len(val_df), "| Test holdout:", len(test_df))
print("Features:", FEATURES)
print("--- VALIDATION METRICS ---")
print(results.round(4).to_string(index=False))
print("Selected model:", selected_name)
print("Model artifact:", model_path)
print("TEST METRICS: NOT RUN | EXTERNAL VALIDATION: NOT RUN")
print("===== KẾT THÚC BƯỚC 5 — GỬI KẾT QUẢ ĐỂ KIỂM TRA =====")

===== BƯỚC 5 — NHANES MODEL TRAINING (NGƯỜI DÙNG TỰ CHẠY) =====
Python: 3.13.9 | Pandas: 3.0.0 | sklearn: 1.8.0
Dataset: 4698 | Target 1: 1387 | Target 0: 3311
Train: 3288 | Validation: 705 | Test holdout: 705
Features: ['RIDAGEYR', 'RIAGENDR', 'BMXBMI']
--- VALIDATION METRICS ---
             model  roc_auc_val  pr_auc_val  precision_val  recall_val  f1_val  accuracy_val  brier_val  tn  fp  fn  tp  train_seconds
LogisticRegression       0.7173      0.4415         0.4416      0.1635  0.2386        0.6922     0.1868 454  43 174  34          0.041
      RandomForest       0.7128      0.4516         0.4886      0.2067  0.2905        0.7021     0.1854 452  45 165  43          0.907
             Dummy       0.5000      0.2950         0.0000      0.0000  0.0000        0.7050     0.2080 497   0 208   0          0.036
Selected model: LogisticRegression
Model artifact: C:\Users\ThinkPad\Multi-Agent_Du_doan_benh_tieu_duong\GĐ3\03-ML-Research\models\baseline\nhanes_hba1c_baseline_step5.joblib
TES